In [1]:
import random
from datetime import datetime, timedelta
import pandas as pd

In [2]:
# Set random seed for reproducibility
random.seed(42)

In [3]:
# Configuration parameters
NUM_PATIENTS = 200
start_date = datetime(2026, 6, 1, 8, 0, 0)

In [4]:
# Reference data pools
genders = ["Male", "Female", "Other"]
doctors = ["Dr. Robert", "Dr. House", "Dr. Smith", "Dr. Adams", "Dr. Chen"]
triage_levels = ["ESI Level 2", "ESI Level 3", "ESI Level 4", "ESI Level 5"]

In [5]:
# Locations mapped to activities
activity_locations = {
    "Check-in": "Reception Desk",
    "Triage": "Triage Room 1",
    "Doctor Consultation": "Consultation Bay B",
    "X-Ray": "Radiology Lab 2",
    "Discharge": "Exit Lobby",
}

In [6]:
resources = {
    "Check-in": "Front Desk Staff",
    "Triage": "Nurse Sarah",
    "Doctor Consultation": "Attending Physician",
    "X-Ray": "Tech Mike",
    "Discharge": "Discharge Coordinator",
}

In [7]:
event_rows = []

In [8]:
for i in range(1, NUM_PATIENTS + 1):
  case_id = f"VISIT-{1000 + i}"
  patient_age = random.randint(18, 85)
  patient_gender = random.choice(genders)
  attending_doctor = random.choice(doctors)
  acuity_level = random.choice(triage_levels)

In [12]:
# 40% chance that a patient is missing paperwork, triggering the rework loop after X-Ray
paperwork_missing_flag = random.random() < 0.40

In [14]:
# Stagger patient arrival times
current_time = start_date + timedelta(
    minutes=random.randint(0, 1440 * 3)
)  # Spread over 3 days

In [16]:
# 1. Check-in
event_rows.append({
    "case_id": case_id,
    "activity": "Check-in",
    "timestamp": current_time,
    "resource": resources["Check-in"],
    "location": activity_locations["Check-in"],
    "acuity_level": acuity_level,
    "patient_age": patient_age,
    "patient_gender": patient_gender,
    "attending_doctor": attending_doctor,
    "paperwork_missing_flag": paperwork_missing_flag,
})

In [18]:
# 2. Triage
current_time += timedelta(minutes=random.randint(5, 20))
event_rows.append({
    "case_id": case_id,
    "activity": "Triage",
    "timestamp": current_time,
    "resource": resources["Triage"],
    "location": activity_locations["Triage"],
    "acuity_level": acuity_level,
    "patient_age": patient_age,
    "patient_gender": patient_gender,
    "attending_doctor": attending_doctor,
    "paperwork_missing_flag": paperwork_missing_flag,
})

In [20]:
# 3. Doctor Consultation
current_time += timedelta(minutes=random.randint(15, 60))
event_rows.append({
    "case_id": case_id,
    "activity": "Doctor Consultation",
    "timestamp": current_time,
    "resource": attending_doctor,
    "location": activity_locations["Doctor Consultation"],
    "acuity_level": acuity_level,
    "patient_age": patient_age,
    "patient_gender": patient_gender,
    "attending_doctor": attending_doctor,
    "paperwork_missing_flag": paperwork_missing_flag,
})

In [22]:
# 4. X-Ray (All patients go to X-Ray in this simulation for simplicity)
current_time += timedelta(minutes=random.randint(10, 30))
event_rows.append({
    "case_id": case_id,
    "activity": "X-Ray",
    "timestamp": current_time,
    "resource": resources["X-Ray"],
    "location": activity_locations["X-Ray"],
    "acuity_level": acuity_level,
    "patient_age": patient_age,
    "patient_gender": patient_gender,
    "attending_doctor": attending_doctor,
    "paperwork_missing_flag": paperwork_missing_flag,
})

In [24]:
# --- INJECTED BOTTLENECK / REWORK LOOP ---
if paperwork_missing_flag:
    # Sent back to Triage due to missing form
    current_time += timedelta(minutes=random.randint(15, 30))
    event_rows.append({
        "case_id": case_id,
        "activity": "Triage",  # Rework step
        "timestamp": current_time,
        "resource": resources["Triage"],
        "location": activity_locations["Triage"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": True,
    })

In [26]:
# Quick re-consultation after form fix
if paperwork_missing_flag:
    current_time += timedelta(minutes=random.randint(10, 25))
    event_rows.append({
        "case_id": case_id,
        "activity": "Doctor Consultation",
        "timestamp": current_time,
        "resource": attending_doctor,
        "location": activity_locations["Doctor Consultation"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": False,
    })

In [28]:
# 5. Discharge
current_time += timedelta(minutes=random.randint(15, 40))
event_rows.append({
    "case_id": case_id,
    "activity": "Discharge",
    "timestamp": current_time,
    "resource": resources["Discharge"],
    "location": activity_locations["Discharge"],
    "acuity_level": acuity_level,
    "patient_age": patient_age,
    "patient_gender": patient_gender,
    "attending_doctor": attending_doctor,
    "paperwork_missing_flag": paperwork_missing_flag,
})

In [29]:
# Convert to DataFrame
df_events = pd.DataFrame(event_rows)

In [30]:
# Sort chronologically per case
df_events = df_events.sort_values(by=["case_id", "timestamp"]).reset_index(
    drop=True
)

In [34]:
# Preview the first 10 rows
print(df_events.head(10))

      case_id             activity           timestamp               resource  \
0  VISIT-1200             Check-in 2026-06-02 05:22:00       Front Desk Staff   
1  VISIT-1200               Triage 2026-06-02 05:27:00            Nurse Sarah   
2  VISIT-1200  Doctor Consultation 2026-06-02 06:08:00              Dr. House   
3  VISIT-1200                X-Ray 2026-06-02 06:32:00              Tech Mike   
4  VISIT-1200            Discharge 2026-06-02 07:09:00  Discharge Coordinator   

             location acuity_level  patient_age patient_gender  \
0      Reception Desk  ESI Level 3           28           Male   
1       Triage Room 1  ESI Level 3           28           Male   
2  Consultation Bay B  ESI Level 3           28           Male   
3     Radiology Lab 2  ESI Level 3           28           Male   
4          Exit Lobby  ESI Level 3           28           Male   

  attending_doctor  paperwork_missing_flag  
0        Dr. House                   False  
1        Dr. House        

In [35]:
# Preview the last 10 rows
print(df_events.tail(10))

      case_id             activity           timestamp               resource  \
0  VISIT-1200             Check-in 2026-06-02 05:22:00       Front Desk Staff   
1  VISIT-1200               Triage 2026-06-02 05:27:00            Nurse Sarah   
2  VISIT-1200  Doctor Consultation 2026-06-02 06:08:00              Dr. House   
3  VISIT-1200                X-Ray 2026-06-02 06:32:00              Tech Mike   
4  VISIT-1200            Discharge 2026-06-02 07:09:00  Discharge Coordinator   

             location acuity_level  patient_age patient_gender  \
0      Reception Desk  ESI Level 3           28           Male   
1       Triage Room 1  ESI Level 3           28           Male   
2  Consultation Bay B  ESI Level 3           28           Male   
3     Radiology Lab 2  ESI Level 3           28           Male   
4          Exit Lobby  ESI Level 3           28           Male   

  attending_doctor  paperwork_missing_flag  
0        Dr. House                   False  
1        Dr. House        

In [33]:
# Save to CSV for Task 1 BigQuery ingestion
df_events.to_csv("careflow_synthetic_ehr_logs.csv", index=False)
print(
    "\nDataset generated successfully! Saved as 'careflow_synthetic_ehr_logs.csv'."
)


Dataset generated successfully! Saved as 'careflow_synthetic_ehr_logs.csv'.
